# Biohub — spatial STIR-Net submission

Frozen leaderboard baseline for repository commit `ac981bf4fa79f9a5f8db2aa24f11adcd87754482`.

This notebook is intentionally thin. The attached **private** inference bundle contains the frozen committed code and inference-only spatial checkpoint. The scored path is:

`spatial STIR-Net → signed multicut → source-instance split-only postfilter → Stage 7/8/10/11 tracking/lineage → submission.csv`

**Learned temporal STIR-Net is disabled for this first leaderboard baseline.**

Security/reproducibility rules for the scored run:
- keep the notebook and inference dataset private;
- keep Kaggle Internet **Off**;
- attach the Biohub competition data and the private `stirnet-biohub-inference-ac981bf4` dataset;
- use a GPU accelerator;
- bundled dependency wheels, when present, are installed with `pip --no-index --no-deps` only.

Run the cells in order. The cell titled **Run the frozen spatial baseline** is the long inference cell.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import zipfile

EXPECTED_REPO_SHA = "ac981bf4fa79f9a5f8db2aa24f11adcd87754482"
COMPETITION_SLUG = "biohub-cell-tracking-during-development"
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
SUBMISSION = WORKING_ROOT / "submission.csv"
WORK_ROOT = WORKING_ROOT / "stirnet_submission_work"

print("Python:", sys.version)
print("Kaggle input root:", INPUT_ROOT)
print("Working root:", WORKING_ROOT)


## Locate and cryptographically verify the private inference bundle

This version supports Kaggle's newer hierarchical mount layout:

`/kaggle/input/datasets/<owner>/<dataset>/...`

Discovery is deliberately bounded to dataset mount directories and **never recursively walks the Biohub OME-Zarr competition tree**.


In [ ]:
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def is_valid_bundle_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "VERSION.txt").is_file()
        and (path / "bundle_manifest.json").is_file()
        and (path / "repo").is_dir()
        and (path / "assets" / "stirnet_spatial.pt").is_file()
    )


def dataset_mount_dirs() -> list[Path]:
    """Return only bounded Kaggle dataset mount roots; never recurse into competition data."""
    mounts: list[Path] = []

    # New Kaggle layout: /kaggle/input/datasets/<owner>/<dataset>/
    datasets_root = INPUT_ROOT / "datasets"
    if datasets_root.is_dir():
        for owner_dir in sorted(datasets_root.iterdir()):
            if not owner_dir.is_dir():
                continue
            for dataset_dir in sorted(owner_dir.iterdir()):
                if dataset_dir.is_dir():
                    mounts.append(dataset_dir)

    # Older flat layout: /kaggle/input/<dataset>/
    if INPUT_ROOT.is_dir():
        for child in sorted(INPUT_ROOT.iterdir()):
            if not child.is_dir() or child.name in {"competitions", "datasets"}:
                continue
            mounts.append(child)

    # Stable deduplication.
    unique: list[Path] = []
    seen: set[str] = set()
    for path in mounts:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique


def direct_bundle_candidates() -> list[Path]:
    candidates: list[Path] = []
    for dataset_dir in dataset_mount_dirs():
        for candidate in (dataset_dir, dataset_dir / "stirnet_kaggle_bundle"):
            if is_valid_bundle_root(candidate):
                candidates.append(candidate.resolve())
    return sorted(set(candidates))


def safe_extract_zip(zip_path: Path, destination: Path) -> None:
    """Extract our private bundle while rejecting absolute/parent-traversal members."""
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as archive:
        for info in archive.infolist():
            member = Path(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError(f"Unsafe ZIP member: {info.filename!r}")
            target = (destination / member).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f"ZIP member escapes extraction root: {info.filename!r}")
        archive.extractall(destination)


def extract_bundle_zip() -> list[Path]:
    # Only inspect immediate files in private dataset mount roots.
    zip_candidates: list[Path] = []
    for dataset_dir in dataset_mount_dirs():
        zip_candidates.extend(sorted(dataset_dir.glob("stirnet_kaggle_bundle.zip")))
        zip_candidates.extend(sorted(dataset_dir.glob("*stirnet*kaggle*bundle*.zip")))

    # Stable deduplication.
    unique: list[Path] = []
    seen: set[str] = set()
    for path in zip_candidates:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(path)
    zip_candidates = unique

    if not zip_candidates:
        return []
    if len(zip_candidates) != 1:
        raise RuntimeError(
            "Expected exactly one attached STIR-Net bundle ZIP, found: "
            f"{zip_candidates}"
        )

    extract_root = WORKING_ROOT / "_stirnet_bundle_extracted"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)

    print("Extracting private bundle:", zip_candidates[0])
    safe_extract_zip(zip_candidates[0], extract_root)

    candidates = []
    for child in extract_root.iterdir():
        if is_valid_bundle_root(child):
            candidates.append(child.resolve())
    if is_valid_bundle_root(extract_root):
        candidates.append(extract_root.resolve())
    return sorted(set(candidates))


bundle_candidates = direct_bundle_candidates()
if not bundle_candidates:
    bundle_candidates = extract_bundle_zip()

if len(bundle_candidates) != 1:
    raise RuntimeError(
        "Could not uniquely locate the private STIR-Net inference bundle. "
        f"Candidates: {bundle_candidates}"
    )

BUNDLE_ROOT = bundle_candidates[0]
REPO_ROOT = BUNDLE_ROOT / "repo"
MANIFEST_PATH = BUNDLE_ROOT / "bundle_manifest.json"
VERSION_PATH = BUNDLE_ROOT / "VERSION.txt"
CHECKPOINT = BUNDLE_ROOT / "assets" / "stirnet_spatial.pt"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

if manifest.get("bundle_type") != "stirnet-kaggle-inference-bundle":
    raise RuntimeError(
        f"Unexpected bundle_type: {manifest.get('bundle_type')!r}"
    )
if manifest.get("source_repo_sha") != EXPECTED_REPO_SHA:
    raise RuntimeError(
        "Bundle repository revision mismatch.\n"
        f"Expected: {EXPECTED_REPO_SHA}\n"
        f"Actual  : {manifest.get('source_repo_sha')}"
    )

version_text = VERSION_PATH.read_text(encoding="utf-8").strip()
if version_text != EXPECTED_REPO_SHA:
    raise RuntimeError(
        "VERSION.txt repository revision mismatch.\n"
        f"Expected: {EXPECTED_REPO_SHA}\nActual  : {version_text}"
    )

files = manifest.get("files")
if not isinstance(files, dict):
    raise RuntimeError(
        "bundle_manifest.json has no valid 'files' mapping. "
        f"Got: {type(files).__name__}"
    )

print("Bundle root :", BUNDLE_ROOT)
print("Repo root   :", REPO_ROOT)
print("Checkpoint  :", CHECKPOINT)
print("Version     :", version_text)
print("Manifest files:", len(files))
print("Verifying SHA256 hashes...")

verified = 0
for relative, record in sorted(files.items()):
    if not isinstance(record, dict):
        raise RuntimeError(f"Malformed manifest record for {relative}: {record!r}")

    expected_hash = record.get("sha256")
    expected_size = record.get("size_bytes")
    if not expected_hash or expected_size is None:
        raise RuntimeError(f"Malformed hash/size record for {relative}: {record!r}")

    path = BUNDLE_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Bundle file is missing: {relative}")

    actual_size = path.stat().st_size
    if actual_size != int(expected_size):
        raise RuntimeError(
            f"Size mismatch for {relative}\n"
            f"expected: {expected_size}\nactual  : {actual_size}"
        )

    actual_hash = sha256_file(path)
    if actual_hash.lower() != str(expected_hash).lower():
        raise RuntimeError(
            f"SHA256 mismatch for {relative}\n"
            f"expected: {expected_hash}\nactual  : {actual_hash}"
        )

    verified += 1
    if verified % 50 == 0 or verified == len(files):
        print(f"  verified {verified}/{len(files)} files")

print()
print("Bundle verification: OK")
print("Source repo SHA     :", manifest["source_repo_sha"])
print("Checkpoint step     :", manifest.get("checkpoint", {}).get("global_step", "unknown"))


## Install bundled offline dependencies and audit the Kaggle environment

If the bundle contains wheels, this cell installs them strictly from the private bundle using `--no-index --no-deps`. It does not enable or use Internet access.

The refreshed Kaggle image seen during preflight did not include `zarr`, so the rebuilt bundle should contain the offline Zarr dependency set.


In [ ]:
wheel_names = manifest.get("offline_wheels", [])
if wheel_names is None:
    wheel_names = []
if not isinstance(wheel_names, list):
    raise RuntimeError("Manifest field 'offline_wheels' must be a list")

wheels = [BUNDLE_ROOT / "wheels" / name for name in wheel_names]
for wheel in wheels:
    if not wheel.is_file():
        raise FileNotFoundError(f"Manifest-listed offline wheel is missing: {wheel}")

if wheels:
    print(f"Installing {len(wheels)} bundled offline wheel(s):")
    for wheel in wheels:
        print("  ", wheel.name)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-deps",
            *map(str, wheels),
        ],
        check=True,
        env={**os.environ, "PIP_DISABLE_PIP_VERSION_CHECK": "1"},
    )
    importlib.invalidate_caches()
    print("Offline dependency installation: OK")
else:
    print("No bundled offline wheels are present.")

required_modules = {
    "torch": "torch",
    "numpy": "numpy",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "pandas": "pandas",
    "zarr": "zarr",
    "networkx": "networkx",
    "tqdm": "tqdm",
}

print()
print("Environment audit:")
missing = []
for module_name, dist_name in required_modules.items():
    try:
        module = importlib.import_module(module_name)
    except ModuleNotFoundError as exc:
        missing.append((module_name, str(exc)))
        print(f"{module_name:12s} MISSING")
        continue

    try:
        version = importlib.metadata.version(dist_name)
    except importlib.metadata.PackageNotFoundError:
        version = getattr(module, "__version__", "unknown")
    print(f"{module_name:12s} {version}")

if missing:
    details = "\n".join(f"  {name}: {err}" for name, err in missing)
    raise RuntimeError(
        "Required Kaggle runtime dependencies are missing after offline installation:\n"
        + details
        + "\nRebuild the private inference bundle with the required Linux/Python 3.12 wheels."
    )

import torch
print()
print("CUDA available :", torch.cuda.is_available())
print("Torch CUDA     :", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Enable a Kaggle GPU accelerator before running inference."
    )
print("GPU            :", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("GPU memory     :", f"{props.total_memory / 2**30:.2f} GiB")

# Small real CUDA execution test.
x = torch.ones((1024, 1024), device="cuda")
y = x @ x
torch.cuda.synchronize()
del x, y
torch.cuda.empty_cache()
print("CUDA smoke test: OK")


## Locate the Biohub competition test mount

This supports the newer `/kaggle/input/competitions/...` layout and the older flat layout. It inspects only immediate test directories; it does not recursively traverse Zarr chunks.

The interactive test set can be only a placeholder. Kaggle's committed scoring rerun can replace it with the hidden test set, so the runtime dynamically discovers samples and does not hard-code IDs, counts, or frame numbers.


In [ ]:
def find_competition_root() -> Path:
    candidates = [
        INPUT_ROOT / "competitions" / COMPETITION_SLUG,
        INPUT_ROOT / COMPETITION_SLUG,
    ]
    for candidate in candidates:
        if (candidate / "test").is_dir():
            return candidate.resolve()

    # Conservative immediate-directory fallback only; no recursive scan.
    if INPUT_ROOT.is_dir():
        for candidate in INPUT_ROOT.iterdir():
            if (
                candidate.is_dir()
                and "biohub" in candidate.name.lower()
                and (candidate / "test").is_dir()
            ):
                return candidate.resolve()

    raise FileNotFoundError("Could not locate the Biohub competition test mount")


COMPETITION_ROOT = find_competition_root()
TEST_ROOT = COMPETITION_ROOT / "test"
CHECKPOINT = BUNDLE_ROOT / manifest["checkpoint"]["bundle_path"]

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Inference checkpoint not found: {CHECKPOINT}")

print("Competition root:", COMPETITION_ROOT)
print("Test root       :", TEST_ROOT)
print("Repo root       :", REPO_ROOT)
print("Checkpoint      :", CHECKPOINT)

# Bounded discovery only: direct or one nesting level.
visible_zarrs = sorted(TEST_ROOT.glob("*.zarr")) + sorted(TEST_ROOT.glob("*/*.zarr"))
# Stable deduplication.
visible_zarrs = list(dict.fromkeys(path.resolve() for path in visible_zarrs))

print()
print("Datasets visible to this execution:", len(visible_zarrs))
for path in visible_zarrs[:50]:
    print("  ", path.name)

if not visible_zarrs:
    raise RuntimeError("No visible .zarr test datasets were found under the competition test mount")

print("Competition preflight: OK")


## Run the frozen spatial baseline

**This is the long-running cell.** Run it only after all previous cells finish successfully.

It deletes only this notebook's previous `/kaggle/working/submission.csv` and `stirnet_submission_work` directory, then executes the frozen bundled runtime. The runtime aborts on any sample failure rather than intentionally producing a partial submission.


In [ ]:
if SUBMISSION.exists():
    SUBMISSION.unlink()
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

runner = REPO_ROOT / "kaggle" / "run_submission.py"
if not runner.is_file():
    raise FileNotFoundError(f"Bundled runner not found: {runner}")

command = [
    sys.executable,
    str(runner),
    "--repo-root", str(REPO_ROOT),
    "--test-root", str(TEST_ROOT),
    "--checkpoint", str(CHECKPOINT),
    "--work-root", str(WORK_ROOT),
    "--submission", str(SUBMISSION),
    "--device", "cuda",
    "--expected-repo-sha", manifest["source_repo_sha"],
]

print("Executing frozen inference command:")
print(" ".join(command))
print()

subprocess.run(
    command,
    check=True,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

if not SUBMISSION.is_file():
    raise RuntimeError("Inference completed without /kaggle/working/submission.csv")

print()
print("Inference complete")
print("submission.csv:", SUBMISSION)
print("submission size:", f"{SUBMISSION.stat().st_size / 2**20:.3f} MiB")


## Independent final validation

The runner validates before returning. This cell repeats validation in a fresh process immediately before the Kaggle output is used for submission.


In [ ]:
validator = REPO_ROOT / "kaggle" / "validate_submission.py"
if not validator.is_file():
    raise FileNotFoundError(f"Bundled validator not found: {validator}")
if not SUBMISSION.is_file():
    raise FileNotFoundError(f"Submission file not found: {SUBMISSION}")

subprocess.run(
    [
        sys.executable,
        str(validator),
        str(SUBMISSION),
        "--test-root", str(TEST_ROOT),
    ],
    check=True,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

import pandas as pd

submission = pd.read_csv(SUBMISSION)
print("Rows    :", len(submission))
print("Datasets:", submission["dataset"].nunique())
print("Nodes   :", int((submission["row_type"] == "node").sum()))
print("Edges   :", int((submission["row_type"] == "edge").sum()))
print()
display(submission.head(10))
print()
print("FINAL OUTPUT:", SUBMISSION)
